In [0]:
dbutils.library.restartPython()

In [0]:
import sys
from datetime import datetime

sys.path.append("../../src")
from dataDP.meta import insert_or_update_process_table
from dataDP.data_management import insert_or_update_table
from dataDP.meta.tables_definitions import src_definitions_metadata, ingest_logs
from dataDP import logger, get_execution_id

In [0]:
start_time = datetime.now()
execution_id = get_execution_id()
logger.info(f"Execution ID: {execution_id}")

In [0]:
# Create a DataFrame from the dataclass
src_def_df = spark.createDataFrame([], src_definitions_metadata.to_schema())
insert_or_update_table(
    df=src_def_df,
    table_name=src_definitions_metadata.table_name,
    key_columns=src_definitions_metadata.key_columns,
    schema_evolution=True,
)

In [ ]:
# Prepare data dictionary for insertion
# with current timestamps for created_at and updated_at, and is_active set to True
data_dict = {
    "source_system": "workspace.taxi.yellow",
    "volume": "/Volumes/workspace/taxi/yellow",
    "select_columns": "VendorID, tpep_pickup_datetime, tpep_dropoff_datetime, passenger_count, trip_distance, RatecodeID, store_and_fwd_flag, PULocationID, DOLocationID, payment_type, fare_amount, extra, mta_tax, tip_amount, tolls_amount, improvement_surcharge, total_amount, congestion_surcharge, Airport_fee, cbd_congestion_fee",
    "is_active": True,
    "created_at": datetime.now(),
    "updated_at": datetime.now(),
    "stg_table": "workspace.stg.yellow_taxi",
}

# Create DataFrame with schema
df = spark.createDataFrame([data_dict], schema=src_definitions_metadata.to_schema())
insert_or_update_table(
    df=df,
    table_name=src_definitions_metadata.table_name,
    key_columns=src_definitions_metadata.key_columns,
    schema_evolution=True,
)

In [ ]:
# Prepare data dictionary for insertion
# without created_at, updated_at and is_active
# these will be handled by the process table function

data_dict = {
    "source_system": "workspace.taxi.green",
    "volume": "/Volumes/workspace/taxi/green",
    "select_columns": "VendorID, lpep_pickup_datetime, lpep_dropoff_datetime, store_and_fwd_flag, RatecodeID, PULocationID, DOLocationID, passenger_count, trip_distance, fare_amount, extra, mta_tax, tip_amount, tolls_amount, ehail_fee, improvement_surcharge, total_amount, payment_type, trip_type, congestion_surcharge, cbd_congestion_fee",
    "stg_table": "workspace.stg.green_taxi",
}

# Create DataFrame with schema
df = spark.createDataFrame([data_dict])
insert_or_update_process_table(
    df=df,
    table_name=src_definitions_metadata.table_name,
    key_columns=src_definitions_metadata.key_columns,
    schema_evolution=True,
)

In [ ]:

# create ingest_logs form a log record
ingest_log_record = {
    "execution_id": execution_id,
    "source_system": "workspace.taxi.green",
    "table_name": "workspace.stg.green_taxi",
    "status": "SUCCESS",
    "records_processed": 1000,
    "records_failed": 0,
    "start_time": start_time,
    "end_time": datetime.now(),
    "duration_min": (datetime.now() - start_time).total_seconds() / 60,
    "message": "Ingestion completed successfully.",
}
ingest_log_df = spark.createDataFrame([ingest_log_record], schema=ingest_logs.to_schema())
insert_or_update_table(
    df=ingest_log_df,
    table_name=ingest_logs.table_name,
    append_only=True,
)